# OECD Main Economic Indicators — Interactive Dashboard

Pulls all four MEI categories from the **free, unauthenticated OECD SDMX REST API**.

**Blocks:**  
- A · Infrastructure (cache + API client)  
- B · Leading Indicators (CLI / BCI / CCI)  
- C · Industrial Production & Prices  
- D · Labour Market  
- E · Financial & Monetary  
- F · Cross-Country Heatmap  
- G · Interactive Widgets  

> **Run all cells in order** (`Cell → Run All`). Live fetches take ~30 s per category due to the 3.5 s rate-limit delay; subsequent runs use the 24-hour file cache and complete in seconds.

In [ ]:
# ── A1 · Imports & Configuration ─────────────────────────────────────────────
import io
import time as _time
import pickle
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")

# ── API constants ──────────────────────────────────────────────────────────────
OECD_BASE  = "https://sdmx.oecd.org/public/rest/data"
START_TIME = "2015"
CACHE_TTL  = 86400      # 24 h
REQUEST_DELAY = 3.5     # seconds between live requests (≤20/min limit)

# ── Country universe ───────────────────────────────────────────────────────────
COUNTRY_LIST = [
    # G7
    "USA", "GBR", "DEU", "FRA", "ITA", "CAN", "JPN",
    # Other major OECD
    "AUS", "AUT", "BEL", "CHE", "DNK", "ESP", "FIN",
    "GRC", "HUN", "IRL", "KOR", "MEX", "NLD", "NOR",
    "NZL", "POL", "PRT", "SWE", "TUR",
    # Aggregates
    "OECD", "EA19", "G7M",
]

COUNTRY_LABELS = {
    "USA": "United States", "GBR": "United Kingdom", "DEU": "Germany",
    "FRA": "France",        "ITA": "Italy",           "CAN": "Canada",
    "JPN": "Japan",         "AUS": "Australia",       "AUT": "Austria",
    "BEL": "Belgium",       "CHE": "Switzerland",     "DNK": "Denmark",
    "ESP": "Spain",         "FIN": "Finland",         "GRC": "Greece",
    "HUN": "Hungary",       "IRL": "Ireland",         "KOR": "Korea",
    "MEX": "Mexico",        "NLD": "Netherlands",     "NOR": "Norway",
    "NZL": "New Zealand",   "POL": "Poland",          "PRT": "Portugal",
    "SWE": "Sweden",        "TUR": "Turkey",
    "OECD": "OECD Total",   "EA19": "Euro Area (19)", "G7M": "G7",
}

COUNTRY_COLORS = {
    "USA": "#1f77b4", "GBR": "#d62728", "DEU": "#2ca02c", "FRA": "#9467bd",
    "ITA": "#8c564b", "CAN": "#e377c2", "JPN": "#bcbd22", "AUS": "#17becf",
    "AUT": "#aec7e8", "BEL": "#ffbb78", "CHE": "#98df8a", "DNK": "#ff9896",
    "ESP": "#c5b0d5", "FIN": "#c49c94", "GRC": "#f7b6d2", "HUN": "#c7c7c7",
    "IRL": "#dbdb8d", "KOR": "#9edae5", "MEX": "#393b79", "NLD": "#637939",
    "NOR": "#8c6d31", "NZL": "#843c39", "POL": "#7b4173", "PRT": "#a55194",
    "SWE": "#ce6dbd", "TUR": "#de9ed6",
    "OECD": "#000000", "EA19": "#444444", "G7M": "#888888",
}

# ── Dataflows ──────────────────────────────────────────────────────────────────
DATAFLOWS = {
    "CLI":      "OECD.SDD.STES,DSD_STES@DF_CLI,4.0",
    "KEI":      "OECD.SDD.STES,DSD_STES@DF_KEI,4.0",
    "STPRI":    "OECD.SDD.STES,DSD_STES@DF_STPRI,4.0",
    "STLABOUR": "OECD.SDD.STES,DSD_STES@DF_STLABOUR,4.0",
    "FINMARK":  "OECD.SDD.STES,DSD_STES@DF_FINMARK,4.0",
}

# ── Subject codes ──────────────────────────────────────────────────────────────
SUBJECT_CODES = {
    "CLI":        "LI",
    "BCI":        "BS",
    "CCI":        "CS",
    "IPI":        "PRINTO01",
    "IPI_MFG":   "PRINTO02",
    "CPI":        "CPALTT01",
    "PPI":        "PIEAMP01",
    "UNEMP":      "LRHUTTTT",
    "RATE_SHORT": "IR3TBB01",
    "RATE_LONG":  "IRLTLT01",
    "EQUITY":     "SPASTT01",
}

G7 = ["USA", "GBR", "DEU", "FRA", "ITA", "CAN", "JPN"]
AGGREGATES = ["OECD", "EA19", "G7M"]

print("✅ A1 · Configuration loaded")
print(f"   Countries: {len(COUNTRY_LIST)}  |  Start: {START_TIME}  |  Cache TTL: {CACHE_TTL//3600}h")


In [ ]:
# ── A2 · File Cache ───────────────────────────────────────────────────────────
CACHE_DIR = Path("./oecd_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _cache_key(url: str) -> str:
    return hashlib.md5(url.encode()).hexdigest()

def cache_get(url: str):
    """Return cached CSV text if fresh, else None."""
    path = CACHE_DIR / f"{_cache_key(url)}.pkl"
    if not path.exists():
        return None
    try:
        with open(path, "rb") as f:
            entry = pickle.load(f)
        if _time.time() - entry["ts"] > CACHE_TTL:
            path.unlink(missing_ok=True)
            return None
        return entry["text"]
    except Exception:
        return None

def cache_set(url: str, text: str) -> None:
    path = CACHE_DIR / f"{_cache_key(url)}.pkl"
    with open(path, "wb") as f:
        pickle.dump({"ts": _time.time(), "text": text}, f)

def cache_clear(older_than_hours: float = 0) -> int:
    """Delete cache entries older than given hours (0 = all)."""
    cutoff = _time.time() - older_than_hours * 3600
    removed = 0
    for p in CACHE_DIR.glob("*.pkl"):
        try:
            with open(p, "rb") as f:
                ts = pickle.load(f).get("ts", 0)
            if ts < cutoff:
                p.unlink()
                removed += 1
        except Exception:
            p.unlink(missing_ok=True)
            removed += 1
    return removed

print(f"✅ A2 · Cache ready  →  {CACHE_DIR.resolve()}")


In [ ]:
# ── A3 · OECD SDMX API Client ────────────────────────────────────────────────
_last_req_t = 0.0

def _build_key(countries, subjects, freq="M"):
    """Build SDMX series key: C1+C2.S1+S2.FREQ"""
    c = "+".join(countries) if isinstance(countries, (list, tuple)) else countries
    s = "+".join(subjects)  if isinstance(subjects,  (list, tuple)) else subjects
    return f"{c}.{s}.{freq}"

def fetch_oecd(dataflow: str, countries, subjects,
               start_time: str = START_TIME, freq: str = "M") -> str:
    """Fetch OECD SDMX data with cache + rate-limit + retry."""
    global _last_req_t
    series_key = _build_key(countries, subjects, freq)
    url = (f"{OECD_BASE}/{dataflow}/{series_key}"
           f"?startTime={start_time}&format=csvfilewithlabels")

    cached = cache_get(url)
    if cached is not None:
        print(f"  💾  cache hit  ·  {dataflow.split('@')[-1].split(',')[0]}  /  {series_key[:50]}…")
        return cached

    # Rate-limit: enforced delay
    elapsed = _time.time() - _last_req_t
    if elapsed < REQUEST_DELAY:
        _time.sleep(REQUEST_DELAY - elapsed)

    for attempt in range(3):
        try:
            resp = requests.get(url, timeout=90,
                                headers={"Accept": "text/csv, application/csv"})
            _last_req_t = _time.time()
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 60))
                print(f"  ⏳  rate-limited — waiting {wait}s …")
                _time.sleep(wait)
                continue
            resp.raise_for_status()
            text = resp.text
            cache_set(url, text)
            df_name = dataflow.split("@")[-1].split(",")[0]
            print(f"  ✅  fetched  ·  {df_name}  /  {series_key[:50]}…")
            return text
        except requests.exceptions.RequestException as e:
            if attempt == 2:
                print(f"  ❌  failed after 3 attempts: {e}")
                return ""
            print(f"  ⚠️   attempt {attempt+1} error: {e}  — retrying …")
            _time.sleep(5)
    return ""

def parse_sdmx_csv(raw_text: str) -> pd.DataFrame:
    """Parse SDMX csvfilewithlabels → MultiIndex(date, country, subject) DF."""
    if not raw_text or len(raw_text.strip()) < 20:
        return pd.DataFrame()

    df = pd.read_csv(io.StringIO(raw_text), low_memory=False)

    # Flexible column detection
    col_map = {}
    for col in df.columns:
        cu = col.upper().replace(" ", "_").replace("-", "_")
        if cu in ("REF_AREA", "REFERENCE_AREA") and "country" not in col_map:
            col_map["country"] = col
        elif cu == "SUBJECT" and "subject" not in col_map:
            col_map["subject"] = col
        elif cu == "TIME_PERIOD" and "date" not in col_map:
            col_map["date"] = col
        elif cu == "OBS_VALUE" and "value" not in col_map:
            col_map["value"] = col

    missing = [k for k in ("country", "subject", "date", "value") if k not in col_map]
    if missing:
        print(f"  ⚠️   parse_sdmx_csv: missing cols {missing}. Available: {list(df.columns[:10])}")
        return pd.DataFrame()

    out = pd.DataFrame({
        "country": df[col_map["country"]].astype(str),
        "subject": df[col_map["subject"]].astype(str),
        "date":    pd.to_datetime(df[col_map["date"]], format="%Y-%m", errors="coerce"),
        "value":   pd.to_numeric(
                       df[col_map["value"]].replace({" ": np.nan, "..": np.nan}),
                       errors="coerce"),
    }).dropna(subset=["date"])

    out = out.set_index(["date", "country", "subject"]).sort_index()
    return out

def pivot_wide(df: pd.DataFrame,
               subject_filter=None,
               country_filter=None) -> pd.DataFrame:
    """Pivot MultiIndex DF → wide (index=date, columns=country)."""
    if df.empty:
        return pd.DataFrame()
    r = df.reset_index()
    if subject_filter:
        sf = [subject_filter] if isinstance(subject_filter, str) else list(subject_filter)
        r = r[r["subject"].isin(sf)]
    if country_filter:
        cf = [country_filter] if isinstance(country_filter, str) else list(country_filter)
        r = r[r["country"].isin(cf)]
    if r.empty:
        return pd.DataFrame()
    pivot = r.pivot_table(index="date", columns="country",
                          values="value", aggfunc="last")
    return pivot.sort_index()

# ── Quick connectivity test ────────────────────────────────────────────────────
def test_connection():
    """Fetch one month of US CLI to verify API connectivity."""
    print("Running connectivity test (USA CLI, 2024) …")
    raw = fetch_oecd(DATAFLOWS["CLI"], ["USA"], ["LI"], start_time="2024")
    if raw:
        df = parse_sdmx_csv(raw)
        print(f"  → Parsed {len(df)} rows. Sample:\n{df.head(3)}")
    else:
        print("  → Empty response. Check connectivity or try again.")

# Uncomment to run:
# test_connection()
print("✅ A3 · API client ready")


---
## Block B · Leading Indicators

In [ ]:
# ── B1 · Fetch Leading Indicators (CLI / BCI / CCI) ──────────────────────────
# Subjects: LI=CLI, BS=BCI, CS=CCI
# NOTE: Not all countries publish all three; aggregates OECD/EA19/G7M
#       are included where available.

print("Fetching Leading Indicators …")

_b_countries = COUNTRY_LIST                          # all 30
_b_subjects  = [SUBJECT_CODES["CLI"],               # LI
                SUBJECT_CODES["BCI"],               # BS
                SUBJECT_CODES["CCI"]]               # CS

_raw_cli = fetch_oecd(DATAFLOWS["CLI"], _b_countries, _b_subjects)
_df_cli_raw = parse_sdmx_csv(_raw_cli)

print(f"\n📊  Leading Indicators raw shape: {_df_cli_raw.shape}")
if not _df_cli_raw.empty:
    print("    Countries found:", sorted(_df_cli_raw.reset_index()["country"].unique().tolist()))
    print("    Subjects found: ", sorted(_df_cli_raw.reset_index()["subject"].unique().tolist()))


In [ ]:
# ── B2 · Parse Leading Indicators ────────────────────────────────────────────
_SUBJ_LI = SUBJECT_CODES["CLI"]   # "LI"
_SUBJ_BS = SUBJECT_CODES["BCI"]   # "BS"
_SUBJ_CS = SUBJECT_CODES["CCI"]   # "CS"

def smooth3m(df: pd.DataFrame) -> pd.DataFrame:
    """3-month centred rolling mean."""
    return df.rolling(3, center=True, min_periods=2).mean()

cli_wide = pivot_wide(_df_cli_raw, subject_filter=_SUBJ_LI)
bci_wide = pivot_wide(_df_cli_raw, subject_filter=_SUBJ_BS)
cci_wide = pivot_wide(_df_cli_raw, subject_filter=_SUBJ_CS)

cli_smooth = smooth3m(cli_wide)
bci_smooth = smooth3m(bci_wide)
cci_smooth = smooth3m(cci_wide)

print(f"✅  CLI wide: {cli_wide.shape}  |  countries: {len(cli_wide.columns)}")
print(f"   BCI wide: {bci_wide.shape}")
print(f"   CCI wide: {cci_wide.shape}")
print("\nLatest CLI values:")
if not cli_wide.empty:
    display(cli_wide.dropna(how="all").tail(3).round(2))


In [ ]:
# ── B3 · Chart — Leading Indicators ──────────────────────────────────────────
def _line_kw(country):
    is_agg = country in AGGREGATES
    return dict(
        line=dict(
            color=COUNTRY_COLORS.get(country, "#aaaaaa"),
            width=2.5 if is_agg else 1.4,
            dash="solid",
        ),
        opacity=0.95 if is_agg else 0.75,
    )

leading_fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=["Composite Leading Indicator (CLI)",
                    "Business Confidence Index (BCI)",
                    "Consumer Confidence Index (CCI)"],
)

_cli_src = cli_smooth if not cli_smooth.empty else cli_wide
_bci_src = bci_smooth if not bci_smooth.empty else bci_wide
_cci_src = cci_smooth if not cci_smooth.empty else cci_wide

for row, (df_w, label) in enumerate([(_cli_src, "CLI"),
                                      (_bci_src, "BCI"),
                                      (_cci_src, "CCI")], start=1):
    if df_w.empty:
        continue
    # Sort: aggregates last (drawn on top)
    cols_sorted = sorted(df_w.columns,
                         key=lambda c: (c in AGGREGATES, c))
    for country in cols_sorted:
        s = df_w[country].dropna()
        if s.empty:
            continue
        kw = _line_kw(country)
        name = COUNTRY_LABELS.get(country, country)
        leading_fig.add_trace(
            go.Scatter(
                x=s.index, y=s.values,
                name=name,
                legendgroup=name,
                showlegend=(row == 1),
                hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}: %{{y:.2f}}<extra></extra>",
                **kw,
            ),
            row=row, col=1,
        )
    # Reference line at 100 for CLI
    if row == 1:
        leading_fig.add_hline(y=100, line_dash="dot",
                               line_color="black", line_width=1,
                               row=row, col=1)

leading_fig.update_layout(
    title="OECD Main Economic Indicators — Leading Indicators",
    height=750,
    legend=dict(
        orientation="v", x=1.01, y=1,
        font=dict(size=10),
        tracegroupgap=0,
    ),
    hovermode="x unified",
    template="plotly_white",
    margin=dict(l=60, r=180, t=60, b=40),
)
leading_fig.update_xaxes(showgrid=True, gridcolor="#eeeeee")
leading_fig.update_yaxes(showgrid=True, gridcolor="#eeeeee", zeroline=False)

leading_fig.show()
print("✅ B3 · Leading Indicators chart rendered")


---
## Block C · Industrial Production & Prices

In [ ]:
# ── C1 · Fetch Industrial Production & Prices ────────────────────────────────
print("Fetching Industrial Production (KEI) …")

_ipi_countries = COUNTRY_LIST
_ipi_subjects  = [SUBJECT_CODES["IPI"], SUBJECT_CODES["IPI_MFG"]]   # PRINTO01, PRINTO02
_raw_kei = fetch_oecd(DATAFLOWS["KEI"], _ipi_countries, _ipi_subjects)
_df_kei_raw = parse_sdmx_csv(_raw_kei)
print(f"   KEI raw shape: {_df_kei_raw.shape}")

print("\nFetching Prices (STPRI) …")
_pri_subjects = [SUBJECT_CODES["CPI"], SUBJECT_CODES["PPI"]]   # CPALTT01, PIEAMP01
_raw_stpri = fetch_oecd(DATAFLOWS["STPRI"], _ipi_countries, _pri_subjects)
_df_stpri_raw = parse_sdmx_csv(_raw_stpri)
print(f"   STPRI raw shape: {_df_stpri_raw.shape}")


In [ ]:
# ── C2 · Parse Industrial Production & Prices ─────────────────────────────────
def yoy(df: pd.DataFrame) -> pd.DataFrame:
    """Year-on-year % change for monthly data."""
    return df.pct_change(12).mul(100).replace([np.inf, -np.inf], np.nan)

# IPI & Manufacturing IPI
ipi_wide     = pivot_wide(_df_kei_raw, subject_filter=SUBJECT_CODES["IPI"])
ipi_mfg_wide = pivot_wide(_df_kei_raw, subject_filter=SUBJECT_CODES["IPI_MFG"])

# Prices
cpi_wide = pivot_wide(_df_stpri_raw, subject_filter=SUBJECT_CODES["CPI"])
ppi_wide = pivot_wide(_df_stpri_raw, subject_filter=SUBJECT_CODES["PPI"])

# Forward-fill max 3 months (bridge minor gaps)
for _df in [ipi_wide, ipi_mfg_wide, cpi_wide, ppi_wide]:
    _df.ffill(limit=3, inplace=True)

# YoY % changes
ipi_yoy = yoy(ipi_wide)
cpi_yoy = yoy(cpi_wide)
ppi_yoy = yoy(ppi_wide)

print(f"✅  IPI  wide: {ipi_wide.shape}  |  YoY: {ipi_yoy.shape}")
print(f"   CPI  wide: {cpi_wide.shape}  |  YoY: {cpi_yoy.shape}")
print(f"   PPI  wide: {ppi_wide.shape}  |  YoY: {ppi_yoy.shape}")

if not ipi_yoy.empty:
    print("\nLatest IPI YoY%:")
    display(ipi_yoy.dropna(how="all").tail(2).round(2))


In [ ]:
# ── C3 · Chart — Industrial Production & Prices ───────────────────────────────
production_fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[
        "Industrial Production Index (2015 = 100)",
        "IPI — Year-on-Year %",
        "CPI & PPI — Year-on-Year %",
    ],
)

# --- Row 1: IPI level ---
for country in (ipi_wide.columns if not ipi_wide.empty else []):
    s = ipi_wide[country].dropna()
    if s.empty:
        continue
    kw = _line_kw(country)
    name = COUNTRY_LABELS.get(country, country)
    production_fig.add_trace(
        go.Scatter(x=s.index, y=s.values, name=name,
                   legendgroup=name, showlegend=True,
                   hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}: %{{y:.1f}}<extra></extra>",
                   **kw),
        row=1, col=1,
    )

# --- Row 2: IPI YoY ---
for country in (ipi_yoy.columns if not ipi_yoy.empty else []):
    s = ipi_yoy[country].dropna()
    if s.empty:
        continue
    kw = _line_kw(country)
    name = COUNTRY_LABELS.get(country, country)
    production_fig.add_trace(
        go.Scatter(x=s.index, y=s.values, name=name,
                   legendgroup=name, showlegend=False,
                   hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}: %{{y:.1f}}%<extra></extra>",
                   **kw),
        row=2, col=1,
    )
production_fig.add_hline(y=0, line_dash="dot", line_color="black",
                          line_width=1, row=2, col=1)

# --- Row 3: CPI & PPI YoY ---
for _df, dash, label_sfx in [(cpi_yoy, "solid", " CPI"), (ppi_yoy, "dash", " PPI")]:
    for country in (_df.columns if not _df.empty else []):
        if country not in G7 + AGGREGATES:
            continue   # Row 3: G7 + aggregates only for clarity
        s = _df[country].dropna()
        if s.empty:
            continue
        name = COUNTRY_LABELS.get(country, country) + label_sfx
        production_fig.add_trace(
            go.Scatter(
                x=s.index, y=s.values, name=name,
                legendgroup=name, showlegend=False,
                line=dict(color=COUNTRY_COLORS.get(country, "#aaaaaa"),
                          width=1.4, dash=dash),
                hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}: %{{y:.1f}}%<extra></extra>",
            ),
            row=3, col=1,
        )
production_fig.add_hline(y=0, line_dash="dot", line_color="black",
                          line_width=1, row=3, col=1)

production_fig.update_layout(
    title="OECD Main Economic Indicators — Industrial Production & Prices",
    height=800,
    legend=dict(orientation="v", x=1.01, y=1, font=dict(size=10)),
    hovermode="x unified",
    template="plotly_white",
    margin=dict(l=60, r=180, t=60, b=40),
)
production_fig.update_xaxes(showgrid=True, gridcolor="#eeeeee")
production_fig.update_yaxes(showgrid=True, gridcolor="#eeeeee", zeroline=False)

production_fig.show()
print("✅ C3 · Industrial Production chart rendered")


---
## Block D · Labour Market

In [ ]:
# ── D1 · Fetch Labour Market ──────────────────────────────────────────────────
print("Fetching Labour Market (STLABOUR) …")

_lab_countries = COUNTRY_LIST
_lab_subjects  = [SUBJECT_CODES["UNEMP"]]   # LRHUTTTT

_raw_lab = fetch_oecd(DATAFLOWS["STLABOUR"], _lab_countries, _lab_subjects)
_df_lab_raw = parse_sdmx_csv(_raw_lab)

print(f"   STLABOUR raw shape: {_df_lab_raw.shape}")
if not _df_lab_raw.empty:
    print("   Countries:", sorted(_df_lab_raw.reset_index()["country"].unique().tolist()))


In [ ]:
# ── D2 · Parse Labour Market ──────────────────────────────────────────────────
unemp_wide = pivot_wide(_df_lab_raw, subject_filter=SUBJECT_CODES["UNEMP"])
unemp_wide.ffill(limit=3, inplace=True)

# Sort countries by latest unemployment rate (descending) for legend ordering
if not unemp_wide.empty:
    _latest_unemp = unemp_wide.iloc[-1].dropna().sort_values(ascending=False)
    _unemp_order  = _latest_unemp.index.tolist()
else:
    _unemp_order = list(unemp_wide.columns)

print(f"✅  Unemployment wide: {unemp_wide.shape}")
if not unemp_wide.empty:
    print("\nLatest unemployment rates (%):")
    display(unemp_wide.dropna(how="all").tail(2).round(2))


In [ ]:
# ── D3 · Chart — Labour Market ────────────────────────────────────────────────
labour_fig = go.Figure()

for country in (_unemp_order if _unemp_order else unemp_wide.columns):
    s = unemp_wide[country].dropna() if country in unemp_wide.columns else pd.Series(dtype=float)
    if s.empty:
        continue
    is_agg  = country in AGGREGATES
    name    = COUNTRY_LABELS.get(country, country)
    labour_fig.add_trace(go.Scatter(
        x=s.index, y=s.values,
        name=name,
        line=dict(
            color=COUNTRY_COLORS.get(country, "#aaaaaa"),
            width=3.0 if country == "OECD" else (2.0 if is_agg else 1.4),
            dash="solid",
        ),
        opacity=0.95 if is_agg else 0.75,
        hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}: %{{y:.1f}}%<extra></extra>",
    ))

labour_fig.update_layout(
    title="OECD Main Economic Indicators — Unemployment Rate (Harmonised, %)",
    height=500,
    legend=dict(orientation="v", x=1.01, y=1, font=dict(size=10),
                tracegroupgap=0),
    hovermode="x unified",
    template="plotly_white",
    margin=dict(l=60, r=180, t=60, b=40),
    yaxis=dict(title="Unemployment Rate (%)", showgrid=True, gridcolor="#eeeeee"),
    xaxis=dict(showgrid=True, gridcolor="#eeeeee"),
)

labour_fig.show()
print("✅ D3 · Labour Market chart rendered")


---
## Block E · Financial & Monetary

In [ ]:
# ── E1 · Fetch Financial & Monetary ──────────────────────────────────────────
print("Fetching Financial & Monetary (FINMARK) …")

_fin_countries = COUNTRY_LIST
_fin_subjects  = [
    SUBJECT_CODES["RATE_SHORT"],  # IR3TBB01 — 3-month rate
    SUBJECT_CODES["RATE_LONG"],   # IRLTLT01 — 10-year yield
    SUBJECT_CODES["EQUITY"],      # SPASTT01 — equity index
]

_raw_fin = fetch_oecd(DATAFLOWS["FINMARK"], _fin_countries, _fin_subjects)
_df_fin_raw = parse_sdmx_csv(_raw_fin)

print(f"   FINMARK raw shape: {_df_fin_raw.shape}")
if not _df_fin_raw.empty:
    print("   Subjects:", sorted(_df_fin_raw.reset_index()["subject"].unique().tolist()))


In [ ]:
# ── E2 · Parse Financial & Monetary ──────────────────────────────────────────
rate_short_wide = pivot_wide(_df_fin_raw, subject_filter=SUBJECT_CODES["RATE_SHORT"])
rate_long_wide  = pivot_wide(_df_fin_raw, subject_filter=SUBJECT_CODES["RATE_LONG"])
equity_wide     = pivot_wide(_df_fin_raw, subject_filter=SUBJECT_CODES["EQUITY"])

# Forward-fill
for _df in [rate_short_wide, rate_long_wide, equity_wide]:
    _df.ffill(limit=3, inplace=True)

# Yield spread = 10Y − 3M  (aligned on common index/columns)
_common_cols = rate_short_wide.columns.intersection(rate_long_wide.columns)
_common_idx  = rate_short_wide.index.intersection(rate_long_wide.index)
yield_spread = (
    rate_long_wide.loc[_common_idx, _common_cols]
    - rate_short_wide.loc[_common_idx, _common_cols]
)

# Equity YoY %
equity_yoy = yoy(equity_wide)

print(f"✅  Short rate wide:  {rate_short_wide.shape}")
print(f"   Long rate wide:   {rate_long_wide.shape}")
print(f"   Yield spread:     {yield_spread.shape}")
print(f"   Equity YoY:       {equity_yoy.shape}")


In [ ]:
# ── E3 · Chart — Financial & Monetary ────────────────────────────────────────
financial_fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[
        "Interest Rates — Short (dashed) & Long (solid)",
        "Yield Spread (10Y − 3M, pp)",
        "Equity Index — Year-on-Year %",
    ],
)

# Row 1: short (dashed) & long (solid) rates
for country in sorted(rate_short_wide.columns,
                       key=lambda c: (c not in G7 + AGGREGATES, c)):
    if country not in G7 + AGGREGATES:
        continue   # limit to G7 + aggregates for readability
    color = COUNTRY_COLORS.get(country, "#aaaaaa")
    name  = COUNTRY_LABELS.get(country, country)
    is_agg = country in AGGREGATES

    for df_r, dash in [(rate_short_wide, "dash"), (rate_long_wide, "solid")]:
        if country not in df_r.columns:
            continue
        s = df_r[country].dropna()
        if s.empty:
            continue
        sfx = " 3M" if dash == "dash" else " 10Y"
        financial_fig.add_trace(
            go.Scatter(
                x=s.index, y=s.values,
                name=name + sfx,
                legendgroup=name,
                showlegend=(dash == "solid"),
                line=dict(color=color,
                          width=2.2 if is_agg else 1.4,
                          dash=dash),
                opacity=0.9 if is_agg else 0.7,
                hovertemplate=f"<b>{name + sfx}</b><br>%{{x|%Y-%m}}: %{{y:.2f}}%<extra></extra>",
            ),
            row=1, col=1,
        )

# Row 2: yield spread (area fill)
for country in yield_spread.columns:
    if country not in G7 + AGGREGATES:
        continue
    s = yield_spread[country].dropna()
    if s.empty:
        continue
    name = COUNTRY_LABELS.get(country, country)
    color = COUNTRY_COLORS.get(country, "#aaaaaa")
    financial_fig.add_trace(
        go.Scatter(
            x=s.index, y=s.values,
            name=name + " spread",
            legendgroup=name,
            showlegend=False,
            fill="tozeroy",
            fillcolor=color.replace(")", ",0.15)").replace("rgb", "rgba")
                        if color.startswith("rgb") else color + "26",
            line=dict(color=color, width=1.4),
            hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}: %{{y:.2f}} pp<extra></extra>",
        ),
        row=2, col=1,
    )
financial_fig.add_hline(y=0, line_dash="dot", line_color="black",
                         line_width=1, row=2, col=1)

# Row 3: equity YoY %
for country in equity_yoy.columns:
    if country not in G7 + AGGREGATES:
        continue
    s = equity_yoy[country].dropna()
    if s.empty:
        continue
    name = COUNTRY_LABELS.get(country, country)
    financial_fig.add_trace(
        go.Scatter(
            x=s.index, y=s.values,
            name=name + " equity",
            legendgroup=name,
            showlegend=False,
            line=dict(color=COUNTRY_COLORS.get(country, "#aaaaaa"), width=1.4),
            hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}: %{{y:.1f}}%<extra></extra>",
        ),
        row=3, col=1,
    )
financial_fig.add_hline(y=0, line_dash="dot", line_color="black",
                         line_width=1, row=3, col=1)

financial_fig.update_layout(
    title="OECD Main Economic Indicators — Financial & Monetary",
    height=800,
    legend=dict(orientation="v", x=1.01, y=1, font=dict(size=10)),
    hovermode="x unified",
    template="plotly_white",
    margin=dict(l=60, r=180, t=60, b=40),
)
financial_fig.update_xaxes(showgrid=True, gridcolor="#eeeeee")
financial_fig.update_yaxes(showgrid=True, gridcolor="#eeeeee", zeroline=False)

financial_fig.show()
print("✅ E3 · Financial & Monetary chart rendered")


---
## Block F · Summary Heatmap

In [ ]:
# ── F1 · Assemble Cross-Country Heatmap Matrix ────────────────────────────────
# 10 indicators × N countries
# Each indicator column is Z-scored independently.
# Unemployment is inverted (higher unemp = worse = red).

HEATMAP_INDICATORS = [
    ("CLI",          cli_wide,        False),   # (label, wide_df, invert)
    ("BCI",          bci_wide,        False),
    ("CCI",          cci_wide,        False),
    ("IPI YoY%",     ipi_yoy,         False),
    ("CPI YoY%",     cpi_yoy,         True),    # high inflation = red
    ("Unemp %",      unemp_wide,      True),    # higher = worse
    ("Rate Short",   rate_short_wide, False),
    ("Rate Long",    rate_long_wide,  False),
    ("Yield Spread", yield_spread,    False),
    ("Equity YoY%",  equity_yoy,      False),
]

# Countries to show (G7 + aggregates + a few others)
HEATMAP_COUNTRIES = G7 + ["AUS", "CHE", "DNK", "KOR", "NLD", "NOR", "SWE"] + AGGREGATES

def last_valid(df: pd.DataFrame, country: str):
    """Return latest non-NaN value for a country column."""
    if df is None or df.empty or country not in df.columns:
        return np.nan
    s = df[country].dropna()
    return float(s.iloc[-1]) if not s.empty else np.nan

raw_matrix = {}
for label, df_w, _ in HEATMAP_INDICATORS:
    raw_matrix[label] = {c: last_valid(df_w, c) for c in HEATMAP_COUNTRIES}

raw_df = pd.DataFrame(raw_matrix, index=HEATMAP_COUNTRIES)

# Z-score per column; invert unemployment & CPI
z_df = raw_df.copy()
for label, _, invert in HEATMAP_INDICATORS:
    col = raw_df[label].dropna()
    if len(col) < 2:
        z_df[label] = 0.0
        continue
    mu, sigma = col.mean(), col.std()
    z_df[label] = (raw_df[label] - mu) / (sigma if sigma > 0 else 1.0)
    if invert:
        z_df[label] *= -1

z_df = z_df.clip(-2.5, 2.5)

print("✅ F1 · Heatmap matrix assembled")
print(f"   Shape: {z_df.shape}  |  Countries: {len(HEATMAP_COUNTRIES)}")
print("\nRaw values:")
display(raw_df.round(2))


In [ ]:
# ── F2 · Chart — Summary Heatmap ──────────────────────────────────────────────
_indicators = list(z_df.columns)
_countries  = list(z_df.index[::-1])   # reversed: first country at top

_z     = z_df.loc[::-1].values.tolist()
_raw   = raw_df.loc[::-1].round(2).values.tolist()

heatmap_fig = go.Figure(go.Heatmap(
    z=_z,
    x=_indicators,
    y=[COUNTRY_LABELS.get(c, c) for c in _countries],
    colorscale="RdBu",
    zmid=0,
    zmin=-2.5,
    zmax=2.5,
    customdata=_raw,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "%{x}: %{customdata} (z-score = %{z:.2f})"
        "<extra></extra>"
    ),
    colorbar=dict(
        title="Z-Score",
        thickness=14,
        len=0.8,
    ),
    xgap=1,
    ygap=1,
))

heatmap_fig.update_layout(
    title="OECD MEI — Cross-Country Heatmap (latest available data, standardised)",
    height=600,
    template="plotly_white",
    xaxis=dict(side="top", tickangle=-30),
    margin=dict(l=120, r=80, t=100, b=40),
    font=dict(size=11),
)

heatmap_fig.show()
print("✅ F2 · Summary heatmap rendered")


---
## Block G · Interactive Dashboard

In [ ]:
# ── G1 · Define Interactive Widgets ──────────────────────────────────────────
# Convert static figures to FigureWidgets (enables in-place updates)
fw_leading    = go.FigureWidget(leading_fig)
fw_production = go.FigureWidget(production_fig)
fw_labour     = go.FigureWidget(labour_fig)
fw_financial  = go.FigureWidget(financial_fig)
fw_heatmap    = go.FigureWidget(heatmap_fig)

# ── Country selector ──────────────────────────────────────────────────────────
_default_countries = [COUNTRY_LABELS.get(c, c) for c in G7 + ["OECD", "EA19"]]
_all_label_options = sorted([COUNTRY_LABELS.get(c, c) for c in COUNTRY_LIST])

country_select = widgets.SelectMultiple(
    options=_all_label_options,
    value=[v for v in _default_countries if v in _all_label_options],
    rows=14,
    description="Countries",
    layout=widgets.Layout(width="200px"),
    style={"description_width": "70px"},
)

# ── Date range slider ──────────────────────────────────────────────────────────
_all_dates = sorted(set(
    list(cli_wide.index)
    + list(ipi_wide.index)
    + list(unemp_wide.index)
    + list(rate_short_wide.index)
))
_date_opts = [d.strftime("%Y-%m") for d in _all_dates if not pd.isnull(d)]
_date_opts = sorted(set(_date_opts))

date_range = widgets.SelectionRangeSlider(
    options=_date_opts,
    index=(0, len(_date_opts) - 1),
    description="Date range",
    continuous_update=False,
    layout=widgets.Layout(width="450px"),
    style={"description_width": "80px"},
)

# ── Smoothing toggle ──────────────────────────────────────────────────────────
smoothing_toggle = widgets.Checkbox(
    value=True,
    description="3M smoothing (CLI/BCI/CCI)",
    indent=False,
    layout=widgets.Layout(width="220px"),
)

# ── Display mode ──────────────────────────────────────────────────────────────
display_mode = widgets.RadioButtons(
    options=["Level", "YoY %"],
    value="Level",
    description="IPI mode:",
    layout=widgets.Layout(width="160px"),
    style={"description_width": "70px"},
)

# ── Tabs ──────────────────────────────────────────────────────────────────────
category_tabs = widgets.Tab(children=[
    fw_leading,
    fw_production,
    fw_labour,
    fw_financial,
    fw_heatmap,
])
for i, title in enumerate(["Leading", "Production", "Labour", "Financial", "Heatmap"]):
    category_tabs.set_title(i, title)

print("✅ G1 · Widgets created")


In [ ]:
# ── G2 · Wire Callbacks & Display ────────────────────────────────────────────
_label_to_code = {v: k for k, v in COUNTRY_LABELS.items()}

def _get_selected_codes():
    return [_label_to_code.get(lbl, lbl) for lbl in country_select.value]

def _date_filter(s: pd.Series, d_start: str, d_end: str) -> pd.Series:
    try:
        return s.loc[d_start:d_end]
    except Exception:
        return s

def update_leading(change=None):
    selected = _get_selected_codes()
    d0, d1   = date_range.value
    use_smooth = smoothing_toggle.value
    src_cli = cli_smooth if use_smooth else cli_wide
    src_bci = bci_smooth if use_smooth else bci_wide
    src_cci = cci_smooth if use_smooth else cci_wide

    with fw_leading.batch_update():
        for trace in fw_leading.data:
            raw_name = trace.name
            # Extract base country label (legendgroup)
            code = _label_to_code.get(raw_name, raw_name)
            visible = raw_name in [COUNTRY_LABELS.get(c, c) for c in selected]
            trace.visible = True if visible else "legendonly"
            # Update x range for date filter (only for visible traces)
            if visible:
                # Determine which df to use
                for src, lbl in [(src_cli, "CLI"), (src_bci, "BCI"), (src_cci, "CCI")]:
                    if code in src.columns:
                        s = _date_filter(src[code].dropna(), d0, d1)
                        if not s.empty:
                            trace.x = s.index
                            trace.y = s.values
                            break

def update_production(change=None):
    selected = _get_selected_codes()
    d0, d1   = date_range.value
    mode     = display_mode.value

    with fw_production.batch_update():
        for trace in fw_production.data:
            raw_name = trace.name
            visible = any(raw_name.startswith(COUNTRY_LABELS.get(c, c)) for c in selected)
            trace.visible = True if visible else "legendonly"

def update_labour(change=None):
    selected = _get_selected_codes()
    d0, d1   = date_range.value
    with fw_labour.batch_update():
        for trace in fw_labour.data:
            visible = any(trace.name.startswith(COUNTRY_LABELS.get(c, c)) for c in selected)
            trace.visible = True if visible else "legendonly"
            if visible:
                code = _label_to_code.get(trace.name, trace.name)
                if code in unemp_wide.columns:
                    s = _date_filter(unemp_wide[code].dropna(), d0, d1)
                    if not s.empty:
                        trace.x = s.index
                        trace.y = s.values

def update_financial(change=None):
    selected = _get_selected_codes()
    with fw_financial.batch_update():
        for trace in fw_financial.data:
            visible = any(trace.name.startswith(COUNTRY_LABELS.get(c, c)) for c in selected)
            trace.visible = True if visible else "legendonly"

# Wire observers
country_select.observe(lambda c: (update_leading(), update_production(),
                                   update_labour(), update_financial()), names="value")
date_range.observe(lambda c: (update_leading(), update_labour()), names="value")
smoothing_toggle.observe(update_leading, names="value")
display_mode.observe(update_production, names="value")

# ── Layout ────────────────────────────────────────────────────────────────────
controls = widgets.VBox([
    widgets.HTML("<b>Filter Controls</b>"),
    widgets.HTML("<hr style='margin:4px 0'>"),
    country_select,
    widgets.HTML("<br>"),
    date_range,
    widgets.HTML("<br>"),
    smoothing_toggle,
    display_mode,
    widgets.HTML(
        "<br><small style='color:#666'>Data: OECD SDMX API<br>"
        "Cache TTL: 24h<br>"
        f"Last loaded: {datetime.now().strftime('%Y-%m-%d %H:%M')}</small>"
    ),
], layout=widgets.Layout(width="240px", padding="8px",
                          border="1px solid #ddd", border_radius="6px"))

dashboard = widgets.HBox(
    [controls, category_tabs],
    layout=widgets.Layout(width="100%"),
)

print("✅ G2 · Dashboard wired — displaying now ↓")
display(dashboard)
